# Python Machine Learning 3rd ed. Colab Smoke Test

這份 notebook 用來確認課程的 Colab runtime、官方 repo clone、常用套件與各章代表性輕量流程可運作。

它不是取代原書 26 份 notebook；它是課前快速檢查。正式上課仍以 `notebooks/colab_ready/` 的分章 notebook 為主。


In [ ]:
import importlib
import os
import platform
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/rasbt/python-machine-learning-book-3rd-edition.git"
REPO_DIR = Path("/content/python-machine-learning-book-3rd-edition") if Path("/content").exists() else Path.cwd() / "_python_ml_3e_official"

def run(cmd):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True)

if not REPO_DIR.exists():
    run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR])

def ensure_import(import_name, pip_name=None):
    pip_name = pip_name or import_name
    if importlib.util.find_spec(import_name) is None:
        run([sys.executable, "-m", "pip", "install", "-q", pip_name])

for import_name, pip_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("scipy", "scipy"),
    ("matplotlib", "matplotlib"),
    ("sklearn", "scikit-learn"),
    ("mlxtend", "mlxtend"),
    ("pyprind", "pyprind"),
    ("flask", "flask"),
    ("wtforms", "wtforms"),
    ("tensorflow", "tensorflow"),
    ("tensorflow_datasets", "tensorflow-datasets"),
    ("gym", "gym==0.26.2"),
]:
    ensure_import(import_name, pip_name)

import numpy as np
if not hasattr(np, "float"):
    np.float = float
if not hasattr(np, "int"):
    np.int = int
if not hasattr(np, "bool8"):
    np.bool8 = np.bool_

print("Python:", sys.version.split()[0], "| Platform:", platform.platform())
versions = {}
for name in ["numpy", "pandas", "scipy", "matplotlib", "sklearn", "tensorflow", "tensorflow_datasets", "gym"]:
    mod = importlib.import_module(name)
    versions[name] = getattr(mod, "__version__", "installed")
versions


In [ ]:
# Verify official notebook inventory.
expected = [
    "ch01/ch01.ipynb", "ch02/ch02.ipynb", "ch03/ch03.ipynb", "ch04/ch04.ipynb",
    "ch05/ch05.ipynb", "ch06/ch06.ipynb", "ch07/ch07.ipynb", "ch08/ch08.ipynb",
    "ch09/ch09.ipynb", "ch10/ch10.ipynb", "ch11/ch11.ipynb", "ch12/ch12.ipynb",
    "ch13/ch13_part1.ipynb", "ch13/ch13_part2.ipynb", "ch13/ch13_part3.ipynb",
    "ch14/ch14_part1.ipynb", "ch14/ch14_part2.ipynb", "ch14/ch14_part3.ipynb",
    "ch15/ch15_part1.ipynb", "ch15/ch15_part2.ipynb", "ch15/downloading-celeba/downloading-celeba.ipynb",
    "ch16/ch16_part1.ipynb", "ch16/ch16_part2.ipynb",
    "ch17/ch17_part1.ipynb", "ch17/ch17_part2.ipynb", "ch18/ch18.ipynb",
]
missing = [p for p in expected if not (REPO_DIR / p).exists()]
assert not missing, missing
print("Official notebook inventory OK:", len(expected))


In [ ]:
# Ch2-Ch7: classic scikit-learn workflows.
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, make_classification
from sklearn.decomposition import PCA
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

iris = load_iris()
X_train, X_test, y_train, y_test = train_test_split(
    iris.data, iris.target, test_size=0.25, random_state=1, stratify=iris.target
)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
clf.fit(X_train, y_train)
pred = clf.predict(X_test)
assert accuracy_score(y_test, pred) > 0.80

pca = PCA(n_components=2).fit_transform(StandardScaler().fit_transform(iris.data))
assert pca.shape == (150, 2)

grid = GridSearchCV(SVC(), {"C": [0.1, 1.0], "kernel": ["linear", "rbf"]}, cv=3)
grid.fit(X_train, y_train)
assert hasattr(grid, "best_params_")

rf = RandomForestClassifier(n_estimators=10, random_state=1).fit(X_train, y_train)
assert rf.predict(X_test).shape == y_test.shape

ada = AdaBoostClassifier(n_estimators=5, random_state=1).fit(X_train, y_train)
assert ada.predict(X_test).shape == y_test.shape

print("Ch2-Ch7 sklearn smoke OK", confusion_matrix(y_test, pred).shape)


In [ ]:
# Ch4: preprocessing API smoke, including OneHotEncoder sparse/sparse_output compatibility.
import inspect
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

df = pd.DataFrame({
    "color": ["green", "red", "blue", None],
    "size": ["M", "L", "XL", "M"],
    "price": [10.1, np.nan, 13.5, 9.2],
})
ohe_kwargs = {"handle_unknown": "ignore"}
if "sparse_output" in inspect.signature(OneHotEncoder).parameters:
    ohe_kwargs["sparse_output"] = False
else:
    ohe_kwargs["sparse"] = False
ct = ColumnTransformer([
    ("cat", OneHotEncoder(**ohe_kwargs), ["color", "size"]),
    ("num", make_pipeline(SimpleImputer(strategy="mean"), StandardScaler()), ["price"]),
])
arr = ct.fit_transform(df)
assert arr.shape[0] == 4
print("Ch4 preprocessing smoke OK", arr.shape)


In [ ]:
# Ch8-Ch11: text, regression, clustering, and web-app imports.
from flask import Flask
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.datasets import make_blobs, make_regression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split

texts = ["great movie", "bad movie", "great acting", "bad acting"]
y = [1, 0, 1, 0]
Xtxt = TfidfVectorizer().fit_transform(texts)
assert Xtxt.shape[0] == 4

Xr, yr = make_regression(n_samples=80, n_features=3, noise=0.1, random_state=1)
X_train, X_test, y_train, y_test = train_test_split(Xr, yr, random_state=1)
reg = LinearRegression().fit(X_train, y_train)
assert r2_score(y_test, reg.predict(X_test)) > 0.90

Xc, _ = make_blobs(n_samples=60, centers=3, random_state=1)
assert KMeans(n_clusters=3, n_init=10, random_state=1).fit_predict(Xc).shape[0] == 60
assert AgglomerativeClustering(n_clusters=3).fit_predict(Xc).shape[0] == 60
assert DBSCAN(eps=1.0).fit_predict(Xc).shape[0] == 60

app = Flask(__name__)
assert app.name == "__main__"
print("Ch8-Ch11 smoke OK")


In [ ]:
# Ch12-Ch17: lightweight TensorFlow/Keras smoke.
import numpy as np
import tensorflow as tf

rng = np.random.default_rng(1)
X = rng.normal(size=(32, 4)).astype("float32")
y = (X[:, 0] + X[:, 1] > 0).astype("int32")

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(4,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(2, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(X, y, epochs=1, batch_size=8, verbose=0)
assert history.history["loss"]

with tf.GradientTape() as tape:
    x = tf.Variable(3.0)
    loss = x * x
grad = tape.gradient(loss, x)
assert float(grad.numpy()) == 6.0

cnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(8, 8, 1)),
    tf.keras.layers.Conv2D(2, 3, activation="relu"),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(2),
])
assert cnn(tf.zeros((2, 8, 8, 1))).shape == (2, 2)

rnn = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(5,)),
    tf.keras.layers.Embedding(20, 4),
    tf.keras.layers.LSTM(4),
    tf.keras.layers.Dense(2),
])
assert rnn(tf.constant([[1, 2, 3, 4, 5]])).shape == (1, 2)

generator = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(3,)),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(2),
])
discriminator = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(4, activation="relu"),
    tf.keras.layers.Dense(1),
])
fake = generator(tf.zeros((2, 3)))
assert discriminator(fake).shape == (2, 1)
print("Ch12-Ch17 TensorFlow smoke OK", tf.__version__)


In [ ]:
# Ch18: Gym new/old step API compatibility smoke.
import gym
import numpy as np

if not getattr(gym, "_pyml3e_old_api_patch", False):
    original_make = gym.make

    class OldStepAPIWrapper(gym.Wrapper):
        def reset(self, *args, **kwargs):
            result = self.env.reset(*args, **kwargs)
            return result[0] if isinstance(result, tuple) and len(result) == 2 else result

        def step(self, action):
            result = self.env.step(action)
            if isinstance(result, tuple) and len(result) == 5:
                obs, reward, terminated, truncated, info = result
                return obs, reward, bool(terminated or truncated), info
            return result

    def patched_make(*args, **kwargs):
        return OldStepAPIWrapper(original_make(*args, **kwargs))

    gym.make = patched_make
    gym._pyml3e_old_api_patch = True

env = gym.make("CartPole-v1")
state = env.reset()
action = env.action_space.sample()
next_state, reward, done, info = env.step(action)
assert np.asarray(state).shape == np.asarray(next_state).shape
print("Ch18 Gym smoke OK")
